# Lecture 1: Introduction to Computer Vision

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 1, with material from Stanford CS231n 2026, Lecture 1.*

Lecture 1 is a tour of the history of computer vision. In this notebook we'll **build small versions of the historical ideas ourselves**, so they're not just names on a slide.

**What you'll learn**
1. What an image looks like to a computer.
2. How a camera turns a 3-D world into a 2-D picture.
3. The big ideas of classic computer vision: edges, grouping, matching, and face detection.
4. The perceptron, why it got stuck, and how hidden layers fixed it.
5. Why AlexNet (2012) changed everything.

Run the cells from top to bottom. Each code cell is short; read the text above it first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage      # for blurring images and computing gradients

## 1. What does a computer see?

To a computer, an image is just a **grid of numbers**. A color image has 3 numbers per pixel (red, green, blue), each from 0 (dark) to 255 (bright).

Let's make a tiny 4×6 image by hand: red on the left, blue on the right.

In [ ]:
height, width = 4, 6
image = np.zeros((height, width, 3), dtype=np.uint8)   # all black to start

image[:, :3] = [220, 40, 40]     # left 3 columns: red
image[:, 3:] = [40, 40, 220]     # right 3 columns: blue

print("shape:", image.shape)     # (height, width, color channels)
print("the red channel:")
print(image[:, :, 0])

plt.imshow(image)
plt.title("what we see")
plt.axis("off")
plt.show()

We see "red and blue". The computer sees 72 numbers. Getting from the numbers to their *meaning* ("that's a cat") is the whole challenge of computer vision. It's called the **semantic gap**.

**Watch out:** image pixels are often stored as `uint8`, which only holds 0 to 255. Going past 255 wraps around to 0, so convert to floats before doing math:

In [ ]:
pixel = np.array([250], dtype=np.uint8)
print("250 + 10 as uint8:", pixel + 10)                    # wraps around!
print("250 + 10 as float:", pixel.astype(float) + 10)

## 2. How a camera makes a picture

The oldest camera is the **camera obscura**: a dark box with a tiny hole. Light from the scene passes through the hole and makes an image on the back wall.

The math is simple. A point at position $(X, Y, Z)$ in front of the camera ($Z$ = distance) lands on the image at

$$x = f\cdot\frac{X}{Z}, \qquad y = f\cdot\frac{Y}{Z}$$

where $f$ (the focal length) controls the zoom. **Dividing by $Z$ is perspective:** things twice as far away look half as big.

In [ ]:
def project(X, Y, Z, f=100):
    # Where a 3-D point lands in the image (pinhole camera).
    return f * X / Z, f * Y / Z

print("point at distance 2:", project(1, 1, 2))
print("same point, twice as far:", project(1, 1, 4), "<- half the size")

Let's use that to draw a cube. A cube has 8 corners; we place it 5 units in front of the camera, shifted a bit sideways so we see it at an angle.

In [ ]:
corners = []
for X in [-1, 1]:
    for Y in [-1, 1]:
        for Z in [-1, 1]:
            corners.append((X + 0.8, Y + 0.6, Z + 5))      # move the cube in front of the camera

# Two corners are joined by an edge if they differ in exactly one coordinate.
edges = []
for i in range(8):
    for j in range(i + 1, 8):
        differences = sum(a != b for a, b in zip(corners[i], corners[j]))
        if differences == 1:
            edges.append((i, j))

plt.figure(figsize=(4, 4))
for i, j in edges:
    x1, y1 = project(*corners[i])
    x2, y2 = project(*corners[j])
    plt.plot([x1, x2], [y1, y2], "k-")
plt.gca().set_aspect("equal")
plt.title("a cube through a pinhole camera")
plt.show()

The back face of the cube looks smaller than the front face: that's perspective. Notice we never "drew a cube": we just applied the projection formula to its corners.

**Try it:** move the cube farther away (change `Z + 5` to `Z + 10`). What happens to the difference between the front and back faces?

## 3. Hubel and Wiesel (1959): how the brain sees

Hubel and Wiesel recorded single neurons in a cat's visual cortex while showing it bars of light. They found:
* **Simple cells** respond to a bar at **one angle** in **one position**.
* **Complex cells** respond to a bar at one angle, **anywhere in a small area**.

This is the blueprint for convolutional networks (Lecture 7): *detect small patterns everywhere, then combine nearby detections*.

A good model of a simple cell is a **Gabor filter**: a striped pattern inside a soft circle. The cell's response is how well the image matches the pattern (multiply pixel by pixel and add up).

In [ ]:
def gabor(angle_degrees, size=15):
    # A striped pattern at the given angle, faded out by a Gaussian.
    angle = np.radians(angle_degrees)
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1]
    across = x * np.cos(angle) + y * np.sin(angle)       # distance across the stripes
    stripes = np.cos(2 * np.pi * across / 8)
    fade = np.exp(-(x**2 + y**2) / (2 * 3**2))
    pattern = stripes * fade
    return pattern - pattern.mean()                       # so a blank patch gives 0

def bar(angle_degrees, shift=0, size=15):
    # A white bar through the middle of a black image, at the given angle.
    angle = np.radians(angle_degrees)
    half = size // 2
    y, x = np.mgrid[-half:half + 1, -half:half + 1]
    across = x * np.cos(angle) + y * np.sin(angle) - shift
    return (np.abs(across) < 2).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))
axes[0].imshow(gabor(0), cmap="gray"); axes[0].set_title("Gabor filter (0°)")
axes[1].imshow(bar(0), cmap="gray"); axes[1].set_title("bar at 0°")
axes[2].imshow(bar(45), cmap="gray"); axes[2].set_title("bar at 45°")
for ax in axes: ax.axis("off")
plt.show()

**A simple cell** = one filter at one place. The response is the sum of (filter × image), and a neuron can't fire negatively, so we take `max(0, response)`.

In [ ]:
def simple_cell(image, angle_degrees):
    response = np.sum(gabor(angle_degrees) * image)
    return max(0, response)

for angle in [0, 30, 60, 90]:
    print(f"bar at {angle:2d}°: simple cell (prefers 0°) responds {simple_cell(bar(angle), 0):5.2f}")

It responds strongly to its favorite angle and not at all to a perpendicular bar. But what if the bar **moves**?

**A complex cell** = many simple cells at nearby positions, keeping the **largest** response (this "take the max" step is called **max pooling**).

In [ ]:
def complex_cell(image, angle_degrees, max_shift=5):
    # Check the simple cell at several shifted positions and keep the strongest response.
    best = 0
    for shift in range(-max_shift, max_shift + 1):
        shifted_image = np.roll(image, shift, axis=1)       # slide the image sideways
        best = max(best, simple_cell(shifted_image, angle_degrees))
    return best

for shift in [0, 2, 4]:
    moved_bar = bar(0, shift=shift)
    print(f"bar moved by {shift}: simple cell {simple_cell(moved_bar, 0):5.2f},  complex cell {complex_cell(moved_bar, 0):5.2f}")

**What we see:** when the bar moves 4 pixels, the simple cell goes silent but the complex cell still fires. Max pooling gives **tolerance to small shifts**. Remember this pattern; CNNs use exactly the same trick.

## 4. Edges (1960s–1980s): from Roberts to Canny

Larry Roberts (1963) built one of the first vision systems: find the **edges** of blocks, then figure out their 3-D shape. An **edge** is where brightness changes quickly, so we can find edges by **subtracting neighboring pixels**.

Let's make a simple "blocks world" image: two gray rectangles on a dark background. We'll also make a noisy copy, because real cameras are noisy.

In [ ]:
blocks = np.zeros((60, 60))
blocks[10:30, 10:35] = 0.6       # one block
blocks[35:55, 25:50] = 0.9       # another block

rng = np.random.default_rng(0)
noisy_blocks = blocks + rng.normal(0, 0.1, blocks.shape)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(blocks, cmap="gray"); axes[0].set_title("clean")
axes[1].imshow(noisy_blocks, cmap="gray"); axes[1].set_title("noisy")
for ax in axes: ax.axis("off")
plt.show()

**Edge detector #1: raw differences.** For each pixel, compare it with its right neighbor and its lower neighbor. Big differences mean an edge.

In [ ]:
def edge_strength(img):
    # How much brightness changes from each pixel to its right and lower neighbors.
    change_right = np.zeros_like(img)
    change_down = np.zeros_like(img)
    change_right[:, :-1] = img[:, 1:] - img[:, :-1]
    change_down[:-1, :] = img[1:, :] - img[:-1, :]
    return np.sqrt(change_right**2 + change_down**2)

true_edges = edge_strength(blocks) > 0.1          # on the clean image, these are the real edges
print("real edge pixels:", true_edges.sum())

To grade a detector, we count two kinds of mistakes: **false alarms** (pixels marked as edges that aren't near a real edge) and **misses** (real edge pixels that weren't found).

In [ ]:
def grade(found_edges):
    near_real_edge = ndimage.binary_dilation(true_edges, iterations=1)   # allow 1 pixel of slack
    false_alarms = np.sum(found_edges & ~near_real_edge)
    misses = np.sum(true_edges & ~ndimage.binary_dilation(found_edges, iterations=1))
    return false_alarms, misses

raw_edges = edge_strength(noisy_blocks) > 0.3
print("raw differences on the noisy image: false alarms = %d, misses = %d" % grade(raw_edges))

Hundreds of false alarms! Noise makes neighboring pixels differ randomly, so every noisy pixel looks like an edge. This is why early systems only worked on clean blocks.

**Edge detector #2: smooth first.** John Canny (1986) designed a better detector. Its first and most important step: **blur the image** to average away the noise, *then* take differences. (Canny's full method adds two more steps: thinning edges to one pixel wide, and keeping weak edges only if they connect to strong ones.)

In [ ]:
smoothed = ndimage.gaussian_filter(noisy_blocks, sigma=1.5)
smooth_edges = edge_strength(smoothed) > 0.1
print("smooth first, then differences:     false alarms = %d, misses = %d" % grade(smooth_edges))

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(noisy_blocks, cmap="gray"); axes[0].set_title("noisy input")
axes[1].imshow(raw_edges, cmap="gray"); axes[1].set_title("raw differences")
axes[2].imshow(smooth_edges, cmap="gray"); axes[2].set_title("smooth, then differences")
for ax in axes: ax.axis("off")
plt.show()

**What we see:** smoothing removes most of the false alarms. (The edges get a bit thicker; that's what Canny's thinning step fixes.)

But even perfect edges don't tell us *what* an object is. The 1966 **MIT Summer Vision Project** hoped to solve vision in one summer; more than 50 years later we're still working on it.

## 5. Grouping (1997): normalized cuts

Another idea: before recognizing objects, **group pixels that belong together**. Shi and Malik (1997) turned this into a graph problem:
* Each pixel is a node.
* Neighboring pixels are connected by a **weight** that is high if their brightness is similar.
* Split the pixels into two groups so that we **cut only weak connections**.

The "normalized" part: dividing by each group's total connections stops the method from cutting off a single lonely pixel.

$$\text{Ncut}(A, B) = \frac{\text{cut}(A, B)}{\text{total weight of }A} + \frac{\text{cut}(A, B)}{\text{total weight of }B}$$

Let's try it on a tiny "image": one row of 10 pixels. The first 4 pixels are an object (a bit brighter than the background), but the lighting gets brighter from left to right, so **no single brightness threshold** separates object from background.

In [ ]:
lighting = np.linspace(0, 0.6, 10)                       # brighter toward the right
is_object = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
row = 0.2 + 0.3 * is_object + lighting
print("pixel brightness:", row.round(2))
print("object pixels:   ", is_object)

Notice: the background on the right (up to 0.80) is brighter than most of the object (0.50 to 0.70). Thresholding brightness will fail. Now let's build the weights between neighboring pixels:

In [ ]:
n = len(row)
weights = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j and abs(i - j) <= 1:                                  # only connect neighbors
            weights[i, j] = np.exp(-(row[i] - row[j])**2 / 0.01)        # similar brightness -> weight near 1

print("weight between pixels 0 and 1 (both object):", weights[0, 1].round(3))
print("weight between pixels 3 and 4 (object/background):", weights[3, 4].round(3))

Now try **every possible split** into two groups (there are only 2¹⁰ = 1024), and keep the one with the smallest normalized cut:

In [ ]:
def ncut(group_a):
    # Normalized cut for splitting the pixels into group_a (True) and the rest (False).
    group_b = ~group_a
    cut = weights[group_a][:, group_b].sum()
    return cut / weights[group_a].sum() + cut / weights[group_b].sum()

best_split, best_value = None, float("inf")
for code in range(1, 2**n - 1):                     # skip "everything in one group"
    group_a = np.array([(code >> k) & 1 for k in range(n)], dtype=bool)
    value = ncut(group_a)
    if value < best_value:
        best_split, best_value = group_a, value

print("normalized cut finds:", best_split.astype(int))
print("true object:         ", is_object)

Compare with the best possible brightness threshold:

In [ ]:
best_threshold_accuracy = 0
for threshold in row:
    guess = (row > threshold).astype(int)
    accuracy = max(np.mean(guess == is_object), np.mean(guess != is_object))
    best_threshold_accuracy = max(best_threshold_accuracy, accuracy)
print(f"best single threshold gets {best_threshold_accuracy:.0%} of pixels right")
print(f"normalized cut gets {max(np.mean(best_split == is_object), np.mean(best_split != is_object)):.0%} right")

**What we see:** grouping uses *"neighbors with similar brightness belong together"*, which isn't fooled by the slow lighting change. (Real images have millions of pixels, so instead of trying every split, Shi and Malik found an approximate answer with an eigenvector. Same idea.)

## 6. Matching (1999): the idea behind SIFT

David Lowe's **SIFT** recognizes objects by matching small patches between images. The challenge: the same patch looks different when the image is **rotated**. SIFT's trick is to describe a patch by its **edge directions**, and to measure those directions relative to the patch's own main direction.

Let's take a patch (a brightness ramp with a square in it) and the same patch rotated by 90°:

In [ ]:
patch = np.tile(np.linspace(0, 1, 16), (16, 1))   # brightness increases to the right: one clear main direction
patch[4:8, 4:8] = 1.0                              # plus a bright square, so it's not just a ramp
rotated = np.rot90(patch)                          # the same thing, rotated 90 degrees

print("raw pixel difference between them:", np.abs(patch - rotated).sum())

fig, axes = plt.subplots(1, 2, figsize=(4, 2))
axes[0].imshow(patch, cmap="gray"); axes[0].set_title("patch")
axes[1].imshow(rotated, cmap="gray"); axes[1].set_title("rotated 90°")
for ax in axes: ax.axis("off")
plt.show()

Comparing raw pixels says these are very different. Now describe each patch by a **histogram of edge directions**: at every pixel, find which way the brightness changes (the gradient direction), and count how many pixels point each way (8 bins of 45°).

In [ ]:
def direction_histogram(p):
    # Count gradient directions in 8 bins, weighted by how strong each edge is.
    grad_y, grad_x = np.gradient(p)
    strength = np.sqrt(grad_x**2 + grad_y**2)
    angle = np.degrees(np.arctan2(grad_y, grad_x)) % 360
    bins = (((angle + 22.5) % 360) // 45).astype(int)    # bin 0 = around 0°, bin 1 = around 45°, ...
    hist = np.zeros(8)
    for b, s in zip(bins.ravel(), strength.ravel()):
        hist[b] += s
    return hist / hist.sum()

print("patch:  ", direction_histogram(patch).round(2))
print("rotated:", direction_histogram(rotated).round(2))

Look closely: the rotated histogram is the **same numbers, shifted over by 2 bins** (2 × 45° = 90°, wrapping around the end), because rotating the patch rotates every edge direction. So if we shift each histogram so that its **biggest bin comes first** (that's "measuring relative to the main direction"), they should match:

In [ ]:
def rotation_invariant(hist):
    main_direction = np.argmax(hist)
    return np.roll(hist, -main_direction)        # rotate the histogram so the main direction is bin 0

difference = np.abs(rotation_invariant(direction_histogram(patch)) - rotation_invariant(direction_histogram(rotated))).sum()
print("difference after aligning main directions:", round(difference, 4))

**What we see:** the descriptor now ignores the rotation. Real SIFT does this for a 4×4 grid of small cells around each keypoint (128 numbers), and also handles changes in scale and brightness. SIFT powered panoramas, 3-D reconstruction, and image recognition through the 2000s.

## 7. Face detection (2001): Viola and Jones

Viola and Jones built one of the first **real-time face detectors**, and one of the first big successes of **machine learning** in vision. Two key ideas:

**Idea 1: simple rectangle features.** "Is the eye region darker than the cheek region below it?" That's just (sum of pixels in one rectangle) − (sum in another).

**Idea 2: the integral image** makes any rectangle sum cost only 4 lookups, which is what made it fast. Entry $(r, c)$ of the integral image holds the sum of everything above and to the left.

In [ ]:
tiny = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 9]])
integral = tiny.cumsum(axis=0).cumsum(axis=1)       # running sums down, then across
print("image:\n", tiny)
print("integral image:\n", integral)

To get the sum of the bottom-right 2×2 block (5 + 6 + 8 + 9 = 28), combine 4 corners of the integral image: take the big total, subtract the strip above and the strip to the left, then add back the corner that got subtracted twice.

In [ ]:
block_sum = integral[2, 2] - integral[0, 2] - integral[2, 0] + integral[0, 0]
print("sum of the bottom-right 2x2 block:", block_sum, "(by hand: 5 + 6 + 8 + 9 = 28)")

Now a tiny experiment: cartoon "faces" (dark eyes, lighter cheeks) vs. random "non-faces". One feature: **(cheek brightness) − (eye brightness)**.

In [ ]:
def make_face(rng):
    face = np.full((12, 12), 0.6)
    face[3:5, 2:5] = 0.2                     # left eye
    face[3:5, 7:10] = 0.2                    # right eye
    face[8:10, 4:8] = 0.3                    # mouth
    return face + rng.normal(0, 0.1, face.shape)

def make_non_face(rng):
    return ndimage.gaussian_filter(rng.random((12, 12)), 1.5)

def eye_feature(img):
    eyes = img[3:5, 2:10].mean()
    cheeks = img[5:7, 2:10].mean()
    return cheeks - eyes                     # faces: cheeks are brighter than eyes

rng = np.random.default_rng(1)
face_scores = [eye_feature(make_face(rng)) for _ in range(200)]
other_scores = [eye_feature(make_non_face(rng)) for _ in range(200)]
print(f"average feature value: faces {np.mean(face_scores):.2f}, non-faces {np.mean(other_scores):.2f}")

Pick the threshold that separates them best:

In [ ]:
best_accuracy, best_threshold = 0, None
for threshold in np.linspace(-0.3, 0.6, 91):
    correct = sum(s > threshold for s in face_scores) + sum(s <= threshold for s in other_scores)
    accuracy = correct / 400
    if accuracy > best_accuracy:
        best_accuracy, best_threshold = accuracy, threshold
print(f"one rectangle feature + a threshold: {best_accuracy:.1%} accuracy (threshold {best_threshold:.2f})")

**What we see:** even one simple feature separates these cartoon faces almost perfectly. Real faces are harder, so Viola and Jones used **AdaBoost** to pick the best few hundred features out of thousands, and a **cascade** that throws away obvious non-faces after checking just one or two features.

**The big lesson of this history:** each era **hand-designed** its features (edges, regions, SIFT, rectangles). Things improved when machine learning chose how to *combine* features, and improved dramatically once networks learned the features *themselves*. That needed lots of data.

## 8. Datasets and the ImageNet challenge

Learning features needs labeled data. **ImageNet** (Deng, Fei-Fei, et al., 2009) has about 15 million images in 22,000 categories. Its yearly challenge (ILSVRC, 2010–2017) used 1,000 categories and scored **top-5 error**: the guess counts as right if the true label is among the model's top 5.

In [ ]:
years  = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
errors = [28.2, 25.8, 16.4, 11.7, 6.7, 3.6, 3.0, 2.3]
colors = ["gray", "gray"] + ["tab:blue"] * 6         # gray = hand-designed features, blue = deep networks

plt.figure(figsize=(7, 3.5))
plt.bar(years, errors, color=colors)
plt.axhline(5.1, color="red", linestyle="--", label="human (about 5.1%)")
plt.ylabel("top-5 error (%)")
plt.title("ImageNet winners (gray: hand-designed features, blue: deep learning)")
plt.legend()
plt.show()

In **2012, AlexNet** (a deep convolutional network trained on GPUs) cut the error from 25.8% to 16.4% in one year. Every winner after that was a deep network.

## 9. A short history of neural networks

### The perceptron (Rosenblatt, 1958)
The perceptron computes a score $w \cdot x + b$ and predicts $+1$ if it's positive, $-1$ otherwise. Its learning rule is simple: **whenever it makes a mistake, nudge the weights toward the right answer**:

$$w \leftarrow w + y\,x, \qquad b \leftarrow b + y$$

Let's trace it by hand on four points.

In [ ]:
points = np.array([[2, 1], [1, 3], [-1, -1], [-2, 1]])
labels = np.array([1, 1, -1, -1])

w = np.zeros(2)
b = 0
for epoch in range(3):
    for x, y in zip(points, labels):
        score = w @ x + b
        if y * score <= 0:                      # wrong side (or exactly on the line)
            w = w + y * x
            b = b + y
            print(f"mistake on {x}: new w = {w}, b = {b}")
print("final predictions:", np.sign(points @ w + b), " true labels:", labels)

### Minsky and Papert (1969): what a perceptron can't do
A perceptron draws a single straight line. **XOR** can't be separated by one line: the points (0,1) and (1,0) are one class, (0,0) and (1,1) the other. Watch the perceptron never finish:

In [ ]:
def perceptron(points, labels, epochs=100):
    w, b = np.zeros(points.shape[1]), 0
    for epoch in range(epochs):
        mistakes = 0
        for x, y in zip(points, labels):
            if y * (w @ x + b) <= 0:
                w, b = w + y * x, b + y
                mistakes += 1
        if mistakes == 0:
            return f"learned it after {epoch + 1} epochs"
    return f"still making mistakes after {epochs} epochs"

xor_points = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
xor_labels = np.array([-1, 1, 1, -1])
print("XOR:", perceptron(xor_points, xor_labels))

This result caused a big loss of interest in neural networks (part of the first "AI winter").

### Backpropagation (1986): adding a hidden layer
With a **hidden layer** between input and output, and **backpropagation** to train all the weights together, XOR becomes easy. Here's a network with 8 hidden units; Lectures 5–6 explain every line of the training loop.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
targets = np.array([0, 1, 1, 0], dtype=float)

rng = np.random.default_rng(0)
W1 = rng.normal(size=(2, 8))     # input -> hidden weights
b1 = np.zeros(8)                 # hidden biases
W2 = rng.normal(size=(8, 1))     # hidden -> output weights
b2 = np.zeros(1)                 # output bias
# Why biases? Without them, the input (0, 0) always gives tanh(0) = 0 and sigmoid(0) = 0.5, whatever the weights.

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for step in range(5000):
    # forward pass: compute the predictions
    hidden = np.tanh(X @ W1 + b1)
    prediction = sigmoid(hidden @ W2 + b2).ravel()
    # backward pass: how should each weight change to reduce the error?
    error = (prediction - targets).reshape(-1, 1) / 4
    grad_W2 = hidden.T @ error
    grad_hidden = (error @ W2.T) * (1 - hidden**2)
    grad_W1 = X.T @ grad_hidden
    # update the weights a little in the right direction
    W1 -= 1.0 * grad_W1
    b1 -= 1.0 * grad_hidden.sum(axis=0)
    W2 -= 1.0 * grad_W2
    b2 -= 1.0 * error.sum(axis=0)

print("predictions:", prediction.round(2), " targets:", targets)

Solved. The hidden layer learns new features in which XOR *is* separable by a line.

### LeNet (1998) and AlexNet (2012)
**LeNet** (LeCun et al.) used backprop to train a convolutional network that read handwritten digits on bank checks. **AlexNet** (Krizhevsky, Sutskever & Hinton, 2012) was a bigger version of the same idea, trained on ImageNet with two GPUs for about a week. Let's count how big it was.

A **convolutional layer** with $C_{out}$ filters, each of size $C_{in}\times k\times k$, has $C_{out}\times(C_{in}\cdot k^2 + 1)$ parameters (the $+1$ is each filter's bias). A **fully connected layer** from $n$ inputs to $m$ outputs has $m\times(n + 1)$.

In [ ]:
# (number of filters, input channels, kernel size) for AlexNet's 5 conv layers
conv_layers = [(64, 3, 11), (192, 64, 5), (384, 192, 3), (256, 384, 3), (256, 256, 3)]
# (inputs, outputs) for its 3 fully connected layers
fc_layers = [(256 * 6 * 6, 4096), (4096, 4096), (4096, 1000)]

conv_params = sum(c_out * (c_in * k * k + 1) for c_out, c_in, k in conv_layers)
fc_params = sum(m * (n + 1) for n, m in fc_layers)
total = conv_params + fc_params
print(f"conv layers: {conv_params:,} parameters")
print(f"fully connected layers: {fc_params:,} parameters")
print(f"total: {total:,}  (about 61 million; {fc_params / total:.0%} are in the fully connected layers)")

**Why 2012?** Three ingredients came together: **algorithms** (backprop, CNNs, plus tricks like ReLU and dropout), **data** (ImageNet), and **computation** (GPUs). None was enough alone.

---
## 10. Computer vision can help and harm

Vision systems now read medical scans, help robots, and monitor patients. They can also cause harm. The **Gender Shades** study (Buolamwini & Gebru, 2018) found commercial face-analysis systems had error rates up to **34.7% for darker-skinned women** but at most **0.8% for lighter-skinned men**.

**Always check accuracy for each group separately**: a good overall number can hide a group that does badly, especially if that group is a small part of the test set.

In [ ]:
# (share of the test set, error rate): illustrative shares, error rates from Gender Shades' extremes
groups = {"lighter-skinned men": (0.50, 0.008), "lighter-skinned women": (0.20, 0.07),
          "darker-skinned men": (0.20, 0.06), "darker-skinned women": (0.10, 0.347)}
overall = sum(share * error for share, error in groups.values())
print(f"overall error: {overall:.1%}  <- looks fine")
for name, (share, error) in groups.items():
    print(f"  {name:22s}: {error:.1%}")

---
## Summary

| Idea | What we built | What it taught |
|---|---|---|
| Images are numbers | a 4×6 image | the semantic gap |
| Pinhole camera | a projected cube | perspective = divide by distance |
| Simple and complex cells | Gabor filter + max over positions | detect patterns, then pool (→ CNNs) |
| Edges | raw vs. smoothed differences | noise breaks naive methods |
| Normalized cuts | a 10-pixel grouping | group by similarity, not brightness |
| SIFT idea | direction histograms | describe patches so rotation doesn't matter |
| Viola–Jones | integral image + one feature | fast features + learning |
| Perceptron | trace + XOR failure | one line isn't enough |
| Hidden layer | XOR solved | learned features |
| AlexNet | parameter count | data + GPUs + algorithms |

### Check your understanding
1. Why does dividing by $Z$ make far-away objects look smaller?
2. What does a complex cell do that a simple cell can't, and what part of a CNN is it like?
3. Why does smoothing help edge detection on noisy images?
4. Why can't a perceptron learn XOR?
5. Where are most of AlexNet's parameters?

<details><summary>Answers</summary>

1. The image position is $f\cdot X/Z$: a bigger $Z$ means a smaller result.
2. It responds to its pattern over a range of positions (it takes the max over nearby simple cells). It's like max pooling.
3. Noise makes neighboring pixels differ randomly; blurring averages the noise away before we take differences.
4. A perceptron's decision boundary is one straight line, and no line separates XOR's two classes.
5. In the fully connected layers (about 96%).
</details>

**Next:** Lecture 2, image classification with nearest neighbors.